In [1]:
import numpy as np
import torch

from rlaopt.atoms import L1Norm, SumSquares
from rlaopt.atoms.polyhedra import Polyhedra
from rlaopt.expression.bilevel_expression import BilevelExpression
from rlaopt.expression.expression import Variable
from rlaopt.operator_split import OperatorSplit
from rlaopt.solvers import ProxGrad, ProxGradConfig, ProxGradStoppingCriteria

In [2]:
A = torch.randn(100, 10)
C = torch.randn(20, 10)
x_fes = torch.randn(
    10,
)
b = A @ x_fes
l = torch.min(C @ x_fes) * torch.ones(20)
u = torch.max(C @ x_fes) * torch.ones(20)

In [3]:
x = Variable(torch.ones(10))

In [4]:
f = 0.5 * ((A @ x - b) ** 2).sum()

In [5]:
f

ProductExpression(
  (exprs): ModuleList(
    (0): ConstExpression()
    (1): UnaryOpExpression(
      (operand): UnaryOpExpression(
        (operand): AddExpression(
          (exprs): ModuleList(
            (0): ProductExpression(
              (exprs): ModuleList(
                (0): ConstExpression()
                (1): Variable(name='var1', id='1', shape=(10,), dtype=torch.float32, device='cpu', requires_grad=True)
              )
            )
            (1): ConstExpression()
          )
        )
        (_operand): AddExpression(
          (exprs): ModuleList(
            (0): ProductExpression(
              (exprs): ModuleList(
                (0): ConstExpression()
                (1): Variable(name='var1', id='1', shape=(10,), dtype=torch.float32, device='cpu', requires_grad=True)
              )
            )
            (1): ConstExpression()
          )
        )
      )
      (_operand): UnaryOpExpression(
        (operand): AddExpression(
          (exprs): Module

In [6]:
P = Polyhedra(x, A=A, b=b, C=C, l=l, u=u)

In [7]:
(A @ x_fes == b).all()

tensor(True)

In [8]:
P.evaluate_at(x=x_fes)

tensor(inf)

In [9]:
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)

In [10]:
# Low-rank matrix completion data generation
Xstar = torch.randn(1000, 10) / (1000**0.5)
Ystar = torch.randn(500, 10) / (500**0.5)
A = Xstar @ Ystar.T

In [11]:
# Initialize variables
X, Y = (
    Variable(torch.randn(1000, 10) / (1000**0.5)),
    Variable(torch.randn(500, 10) / (500**0.5)),
)

In [12]:
# Setup the objective function
obj = SumSquares(X @ Y.T - A)

In [13]:
# Test objective function evaluation
obj_val = obj.forward()
obj_true = torch.linalg.norm(X.value @ Y.value.T - A, ord="fro") ** 2
print("Error in computed Objective value:", abs(obj_val - obj_true))

Error in computed Objective value: tensor(4.9738e-14, grad_fn=<AbsBackward0>)


In [14]:
# Setup APG with linesearch
config = ProxGradConfig(eta=1.0, use_acceleration=True, use_linesearch=True)
opt = ProxGrad(obj, config)

In [15]:
# Solve with APG step method
params = obj.params
state = opt.init_state(params)
for i in range(200):
    params, state = opt.step(params, state)
    if i % 10 == 0:
        print(f"Iteration {i}, Objective Value: {obj.evaluate(params)}")
print(state.err)

Iteration 0, Objective Value: 13.413015693276039
Iteration 10, Objective Value: 1.9422768506441588
Iteration 20, Objective Value: 0.004236197198553045
Iteration 30, Objective Value: 0.00010528640617145362
Iteration 40, Objective Value: 2.3994600695545825e-06
Iteration 50, Objective Value: 8.084793364853947e-07
Iteration 60, Objective Value: 6.663758379141637e-08
Iteration 70, Objective Value: 1.3330972016066017e-09
Iteration 80, Objective Value: 6.82097523136684e-10
Iteration 90, Objective Value: 1.7305135137355505e-10
Iteration 100, Objective Value: 1.2648250642784545e-11
Iteration 110, Objective Value: 3.9798223722447415e-13
Iteration 120, Objective Value: 3.706143411081748e-13
Iteration 130, Objective Value: 8.126673290194967e-14
Iteration 140, Objective Value: 4.109334777689724e-15
Iteration 150, Objective Value: 3.4278568862195573e-16
Iteration 160, Objective Value: 2.933806773724065e-16
Iteration 170, Objective Value: 4.813263827907348e-17
Iteration 180, Objective Value: 1.617342

In [16]:
names = list(params.keys())

In [17]:
torch.linalg.norm(params[names[0]] @ params[names[1]].T - A, ord="fro") ** 2

tensor(2.5687e-19, grad_fn=<PowBackward0>)

In [18]:
# Lasso data generation
n, p = 1024, 128
ntst = 256
s = 32
J = np.random.choice(p, s)

xStar = torch.zeros(p)
xStar[J] = torch.randn(s) / (s**0.5)
A = torch.randn(n, p) / (n**0.5)
Atst = torch.randn(ntst, p) / (ntst**0.5)
b = A @ xStar + 0.001 * torch.randn(n)
btst = Atst @ xStar + 0.001 * torch.randn(ntst)

In [19]:
# Init params + reg
x = Variable(torch.zeros(p))
mu = 0.1 * torch.linalg.norm(A.T @ b, ord=torch.inf)

In [20]:
# Lasso problem
F = SumSquares(A @ x - b) + L1Norm(x, scaling=mu)

In [21]:
f, r = F.operator_split()

In [22]:
F.forward()

tensor(0.7463, grad_fn=<SumBackward1>)

In [23]:
torch.linalg.norm(A @ x.value - b, 2) ** 2 + mu * torch.linalg.norm(x.value, ord=1)

tensor(0.7463, grad_fn=<AddBackward0>)

In [24]:
# Step size is reciprocal of Lipschitz constant
eta = 1 / (2 * torch.linalg.norm(A, ord=2) ** 2)

In [25]:
config = ProxGradConfig(eta=eta, use_acceleration=True, use_linesearch=False)
opt = ProxGrad(F, config)

In [26]:
# Solve the problem using step method
params = F.params
state = opt.init_state(params)
for i in range(100):
    print(state.err)
    params, state = opt.step(params, state)
print(state.err)

inf
tensor(0.6530, grad_fn=<DivBackward0>)
tensor(0.2503, grad_fn=<DivBackward0>)
tensor(0.0459, grad_fn=<DivBackward0>)
tensor(0.0345, grad_fn=<DivBackward0>)
tensor(0.0329, grad_fn=<DivBackward0>)
tensor(0.0169, grad_fn=<DivBackward0>)
tensor(0.0051, grad_fn=<DivBackward0>)
tensor(0.0034, grad_fn=<DivBackward0>)


tensor(0.0034, grad_fn=<DivBackward0>)
tensor(0.0021, grad_fn=<DivBackward0>)
tensor(0.0009, grad_fn=<DivBackward0>)
tensor(0.0005, grad_fn=<DivBackward0>)
tensor(0.0005, grad_fn=<DivBackward0>)
tensor(0.0003, grad_fn=<DivBackward0>)
tensor(0.0002, grad_fn=<DivBackward0>)
tensor(0.0001, grad_fn=<DivBackward0>)
tensor(7.9520e-05, grad_fn=<DivBackward0>)
tensor(5.2357e-05, grad_fn=<DivBackward0>)
tensor(3.4249e-05, grad_fn=<DivBackward0>)
tensor(2.5550e-05, grad_fn=<DivBackward0>)
tensor(1.7150e-05, grad_fn=<DivBackward0>)
tensor(9.5354e-06, grad_fn=<DivBackward0>)
tensor(6.8390e-06, grad_fn=<DivBackward0>)
tensor(6.2898e-06, grad_fn=<DivBackward0>)
tensor(4.4980e-06, grad_fn=<DivBackward0>)
tensor(2.2397e-06, grad_fn=<DivBackward0>)
tensor(1.2709e-06, grad_fn=<DivBackward0>)
tensor(1.4687e-06, grad_fn=<DivBackward0>)
tensor(1.2499e-06, grad_fn=<DivBackward0>)
tensor(6.9975e-07, grad_fn=<DivBackward0>)
tensor(2.6232e-07, grad_fn=<DivBackward0>)
tensor(3.0729e-07, grad_fn=<DivBackward0>)


In [27]:
params

{'exprs.0.x.exprs.0.exprs.1.value': tensor([ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000, -0.2441,  0.2680,
         -0.0044,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000, -0.0237,  0.0000,  0.0000,  0.0000,  0.0698,
          0.0000,  0.0000, -0.2120,  0.0080,  0.0000,  0.0000,  0.0000, -0.0756,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         -0.1516, -0.0709,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.1635,
          0.0000,  0.0000,  0.0547,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000, -0.0173,  0.0000,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000, -0.1364,
          0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
          0.0000,  0.0000, -0.2117,  0.0000,  0.2477,  0.0000,  0.0000,  0.0000,
          0.1438,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0

In [28]:
# Expected to be the same as at initialization, since the optimizer does not update in-place
x.value

Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0.], requires_grad=True)

In [29]:
# Test OperatorSplit interface with the same problem
obj = OperatorSplit(SumSquares(A @ x - b), L1Norm(x, scaling=mu))

In [30]:
config = ProxGradConfig(eta=eta, use_acceleration=True, use_linesearch=False)
opt = ProxGrad(obj, config)

In [31]:
# Solve the problem using step method
params = obj.f.params
state = opt.init_state(params)
for i in range(100):
    print(state.err)
    params, state = opt.step(params, state)
print(state.err)

inf
tensor(0.6530, grad_fn=<DivBackward0>)
tensor(0.2503, grad_fn=<DivBackward0>)
tensor(0.0459, grad_fn=<DivBackward0>)
tensor(0.0345, grad_fn=<DivBackward0>)
tensor(0.0329, grad_fn=<DivBackward0>)
tensor(0.0169, grad_fn=<DivBackward0>)
tensor(0.0051, grad_fn=<DivBackward0>)
tensor(0.0034, grad_fn=<DivBackward0>)
tensor(0.0034, grad_fn=<DivBackward0>)
tensor(0.0021, grad_fn=<DivBackward0>)
tensor(0.0009, grad_fn=<DivBackward0>)
tensor(0.0005, grad_fn=<DivBackward0>)
tensor(0.0005, grad_fn=<DivBackward0>)
tensor(0.0003, grad_fn=<DivBackward0>)
tensor(0.0002, grad_fn=<DivBackward0>)
tensor(0.0001, grad_fn=<DivBackward0>)
tensor(7.9520e-05, grad_fn=<DivBackward0>)
tensor(5.2357e-05, grad_fn=<DivBackward0>)
tensor(3.4249e-05, grad_fn=<DivBackward0>)
tensor(2.5550e-05, grad_fn=<DivBackward0>)
tensor(1.7150e-05, grad_fn=<DivBackward0>)
tensor(9.5354e-06, grad_fn=<DivBackward0>)
tensor(6.8390e-06, grad_fn=<DivBackward0>)
tensor(6.2898e-06, grad_fn=<DivBackward0>)
tensor(4.4980e-06, grad_fn=<

In [32]:
# Test HVP computation
v = torch.randn(p)
Hv = obj.hvp_f(params, v)
print("HVP error:", torch.linalg.norm(Hv - 2 * (A.T @ (A @ v))))

HVP error: tensor(0., grad_fn=<LinalgVectorNormBackward0>)


In [33]:
# Bilevel Expression for optimizing lasso regularization parameter

# Inner function: Lasso on training data
ftr = lambda mu: SumSquares(A @ x - b) + L1Norm(x, mu)

# Outer function: Least squares on test data
ftst = SumSquares(Atst @ x - btst)

# Setup bilevel problem
obj = BilevelExpression(mu, ftr, ftst, config, ProxGradStoppingCriteria(), ProxGrad)

In [34]:
# Setup solver for the bilevel problem
config_blvl = ProxGradConfig(
    eta=1e-3,
    use_acceleration=False,
    use_linesearch=False,
)
opt_blvl = ProxGrad(obj, config_blvl)

In [35]:
# Initial objective value
obj.forward()

tensor(0.0105, grad_fn=<SumBackward0>)

In [36]:
# # Solve the bilevel problem using solve method
mu_star, err = opt_blvl.solve()

In [37]:
# Final objective value
obj.evaluate(mu_star)

tensor(0.0010, grad_fn=<SumBackward0>)

In [38]:
# Print the optimal regularization parameter
mu_star

{'w': tensor(0.0097, grad_fn=<SubBackward0>)}

In [39]:
# Print gradient norm of the objective at mu_star
torch.func.grad(obj.evaluate)(mu_star)["w"].norm().item()

0.14299270702927053